<a href="https://colab.research.google.com/github/hannaginther/ENGG680_2025_Fall/blob/main/Project/CatBoost_Clf_Regression_Models_200k.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [44]:
!pip install catboost

In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, cohen_kappa_score,
                             mean_squared_error, mean_absolute_error, r2_score)
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================================
# LOAD DATA
# ============================================================================
print("="*60)
print("LOADING DATA")
print("="*60)

df = pd.read_feather('/content/drive/MyDrive/Sparcs_Datafiles/model_df_clf_v2.feather')

print(f"Loaded {len(df):,} rows")
print(f" Columns: {len(df.columns)}")
print(df.head())

# ============================================================================
# 0. STRATIFIED SAMPLING (200k from 4M rows)
# ============================================================================

print("="*60)
print("CREATING STRATIFIED SAMPLING...")
print("="*60)

# Convert category to numeric
category_mapping = {'Short': 0, 'Moderate': 1, 'Long': 2, 'Extreme': 3}
if df['los_category'].dtype == 'object':
    df['los_category_numeric'] = df['los_category'].map(category_mapping)
else:
    df['los_category_numeric'] = df['los_category']

print(f"\nOriginal dataset: {len(df):,} rows")
print("\nOriginal category distribution:")
category_counts = df['los_category_numeric'].value_counts().sort_index()
print(category_counts)
print("\nProportions:")
print(df['los_category_numeric'].value_counts(normalize=True).sort_index())

# Check if we have all categories
available_categories = df['los_category_numeric'].unique()
print(f"\nAvailable categories: {sorted(available_categories)}")

# Stratified sample to maintain category distribution
SAMPLE_SIZE = 200_000

# Method 1: Use sklearn's train_split for stratified sampling
from sklearn.model_selection import train_test_split

# Calculated sample size we actually get
max_possible_sample = min(SAMPLE_SIZE, len(df))

# Use stratified sampling with sklearn
df_sample, _ = train_test_split(
    df,
    train_size=max_possible_sample,
    stratify=df['los_category_numeric'],
    random_state=42
)

print(f"\nSample dataset: {len(df_sample):,} rows")
print("\nSample category distribution:")
sample_counts = df_sample['los_category_numeric'].value_counts().sort_index()
print(sample_counts)
print("\nProportions:")
print(df_sample['los_category_numeric'].value_counts(normalize=True).sort_index())

# Verify distribution is maintained
print("\nDistribution comparison:")
print(f"{'Category':<12} {'Original %':<15} {'Sample %':<15} {'Original Count':<18} {'Sample Count':<15}")
print("-" * 85)
for cat in sorted(available_categories):
    orig_pct = (df['los_category_numeric'] == cat).mean() * 100
    sample_pct = (df_sample['los_category_numeric'] == cat).mean() * 100
    orig_count = (df['los_category_numeric'] == cat).sum()
    sample_count = (df_sample['los_category_numeric'] == cat).sum()
    print(f"{cat:<12} {orig_pct:<15.2f} {sample_pct:<15.2f} {orig_count:<18,} {sample_count:<15,}")

# Check for minimum samples per category
min_samples_per_cat = df_sample['los_category_numeric'].value_counts().min()
print(f"\nMinimum samples in any category: {min_samples_per_cat}:,")

if min_samples_per_cat < 100:
  print("WARNING: some categories have fewer than 100 samples.")
else:
  print("All categories have at least 100 samples")

# Use sampled data for rest of pipeline
df=df_sample.copy()

LOADING DATA
Loaded 4,238,636 rows
 Columns: 20
  health_service_area hospital_county facility_id age_group zip_code gender  \
0       New York City           Bronx        3058     50-69      104      F   
1       New York City           Bronx        1168     30-49      104      M   
2       New York City           Bronx        3058     50-69      104      M   
3       New York City           Bronx        1169     18-29      104      M   
4       New York City           Bronx        1169     50-69      104      F   

                     race          ethnicity  length_of_stay admission_type  \
0              Other Race   Spanish/Hispanic               1      Emergency   
1  Black/African American  Not Span/Hispanic               4      Emergency   
2              Other Race  Not Span/Hispanic               4      Emergency   
3  Black/African American  Not Span/Hispanic               5      Emergency   
4              Other Race   Spanish/Hispanic               3      Emergency   

  

In [ ]:
# ============================================================================
# 1. DATA PREPARATION
# ============================================================================

# Define categorical features (all except length_of_stay and num_payment_types)
cat_features = [
    'health_service_area', 'hospital_county', 'facility_id', 'age_group',
    'zip_code', 'gender', 'race', 'ethnicity', 'admission_type',
    'ccsr_dx_code', 'ccsr_px_code', 'apr_drg_code', 'apr_mdc_code',
    'apr_severity_code', 'apr_mortality_risk', 'apr_med_surg_desc',
    'payment_type'
]

# Feature columns (exclude target variables)
feature_cols = cat_features + ['num_payment_types']

# Prepare X and y
X = df[feature_cols].copy()
y_days = df['length_of_stay'].copy()
y_category = df['los_category_numeric'].copy()

# Check distribution
print("\n" + "="*60)
print("TARGET VARIABLE DISTRIBUTION")
print("="*60)
print("\nLength of Stay Statistics: ")
print(y_days.describe())
print("\nLength of Stay Distribution: ")
print(y_category.value_counts(normalize=True).sort_index())
print(f"\nTotal samples: {len(df):,}")

# Train-test split with stratification
X_train, X_test, y_cat_train, y_cat_test, y_days_train, y_days_test = train_test_split(
    X,
    y_category,
    y_days,
    test_size=0.2,
    stratify=y_category,
    random_state=42
)

print(f"\nTraining samples: {len(X_train):,}")
print(f"Testing samples: {len(X_test):,}")


TARGET VARIABLE DISTRIBUTION

Length of Stay Statistics: 
count    200000.00000
mean          5.77725
std           8.69489
min           1.00000
25%           2.00000
50%           3.00000
75%           6.00000
max         120.00000
Name: length_of_stay, dtype: float64

Length of Stay Distribution: 
los_category_numeric
0    0.391415
1    0.317615
2    0.163970
3    0.127000
Name: proportion, dtype: float64

Total samples: 200,000

Training samples: 160,000
Testing samples: 40,000


In [ ]:
# ============================================================================
# 2. STAGE 1: CLASSIFIER
# ============================================================================
print("\n" + "="*60)
print("STAGE 1: TRAINING CLASSIFIER")
print("="*60)

# Create pools
train_pool_clf = Pool(
    data=X_train,
    label=y_cat_train,
    cat_features=cat_features
)

test_pool_clf = Pool(
    data=X_test,
    label=y_cat_test,
    cat_features=cat_features
)

# Classifier with parameters optimized for imbalanced ordinal data
classifier = CatBoostClassifier(
    iterations=500,  # Reduced from 2000 for faster training
    learning_rate=0.1,  # Increased to converge faster
    depth=6,  # Reduced from 10 for speed

    # Critical for high-cardinality categorical features
    cat_features=cat_features,
    max_ctr_complexity=3,  # Reduced from 5 for speed

    # Handle class imbalance - THIS IS KEY!
    auto_class_weights='Balanced',

    # Regularization
    l2_leaf_reg=3,
    random_strength=1,

    # Sampling for better minority class learning
    bootstrap_type='Bernoulli',
    subsample=0.66,  # More aggressive subsampling for speed

    # Metric for ordinal classification
    eval_metric='WKappa',

    # Performance - using CPU
    task_type='CPU',
    thread_count=-1,

    early_stopping_rounds=50,  # Stop earlier if not improving
    verbose=50,
    random_state=42
)

# Train classifier
print("\nTraining classifier...")
classifier.fit(train_pool_clf, eval_set=test_pool_clf, plot=False)

# Evaluate classifier
y_cat_pred = classifier.predict(X_test)
y_cat_probs = classifier.predict_proba(X_test)

print("\n" + "-"*60)
print("CLASSIFIER PERFORMANCE")
print("-"*60)

kappa = cohen_kappa_score(y_cat_test, y_cat_pred, weights='quadratic')
print(f'Quadratic Weighted Kappa: {kappa:.3f}')
print(f'Accuracy: {(y_cat_test == y_cat_pred).mean():.3f}')

print("\nClassification Report:")
print(classification_report(y_cat_test, y_cat_pred,
                           target_names=['Short', 'Moderate', 'Long', 'Extreme'],
                           digits=3))

# Confusion matrix
cm = pd.crosstab(y_cat_test, y_cat_pred,
                 rownames=['Actual'], colnames=['Predicted'],
                 normalize='index')
print("\nConfusion Matrix (Row-Normalized):")
print(cm.round(3))


STAGE 1: TRAINING CLASSIFIER

Training classifier...
0:	learn: 0.6149248	test: 0.6154954	best: 0.6154954 (0)	total: 3.75s	remaining: 31m 9s
50:	learn: 0.6733359	test: 0.6784903	best: 0.6784903 (50)	total: 3m 5s	remaining: 27m 13s
100:	learn: 0.6806695	test: 0.6857450	best: 0.6857450 (100)	total: 5m 29s	remaining: 21m 42s
150:	learn: 0.6853554	test: 0.6902245	best: 0.6902245 (150)	total: 8m 53s	remaining: 20m 32s
200:	learn: 0.6884618	test: 0.6917634	best: 0.6922690 (196)	total: 12m 17s	remaining: 18m 16s
250:	learn: 0.6909941	test: 0.6932003	best: 0.6936064 (236)	total: 15m 13s	remaining: 15m 6s
300:	learn: 0.6928591	test: 0.6930833	best: 0.6937404 (281)	total: 17m 31s	remaining: 11m 35s
350:	learn: 0.6945569	test: 0.6939214	best: 0.6946123 (330)	total: 20m 6s	remaining: 8m 32s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.6946122935
bestIteration = 330

Shrink model to first 331 iterations.

------------------------------------------------------------
CLASSIFIER

ValueError: Data must be 1-dimensional, got ndarray of shape (40000, 40000) instead

In [ ]:
# Evaluate classifier (FIXED VERSION)
y_cat_pred = classifier.predict(X_test)
y_cat_probs = classifier.predict_proba(X_test)

# Convert to numpy arrays to avoid pandas issues
y_cat_test_array = np.array(y_cat_test)
y_cat_pred_array = np.array(y_cat_pred).ravel()

print("\n" + "-"*60)
print("CLASSIFIER PERFORMANCE")
print("-"*60)

kappa = cohen_kappa_score(y_cat_test_array, y_cat_pred_array, weights='quadratic')
print(f'Quadratic Weighted Kappa: {kappa:.3f}')
print(f'Accuracy: {(y_cat_test_array == y_cat_pred_array).mean():.3f}')

print("\nClassification Report:")
print(classification_report(y_cat_test_array, y_cat_pred_array,
                           target_names=['Short', 'Moderate', 'Long', 'Extreme'],
                           digits=3))

# Confusion matrix
cm = pd.crosstab(y_cat_test_array, y_cat_pred_array,
                 rownames=['Actual'], colnames=['Predicted'],
                 normalize='index')
print("\nConfusion Matrix (Row-Normalized):")
print(cm.round(3))

print("\nClassifier evaluation complete. Continue to Stage 2...")


------------------------------------------------------------
CLASSIFIER PERFORMANCE
------------------------------------------------------------
Quadratic Weighted Kappa: 0.651
Accuracy: 0.544

Classification Report:
              precision    recall  f1-score   support

       Short      0.717     0.663     0.689     15656
    Moderate      0.491     0.395     0.438     12705
        Long      0.330     0.445     0.379      6559
     Extreme      0.531     0.674     0.594      5080

    accuracy                          0.544     40000
   macro avg      0.517     0.544     0.525     40000
weighted avg      0.558     0.544     0.546     40000


Confusion Matrix (Row-Normalized):
Predicted      0      1      2      3
Actual                               
0          0.663  0.218  0.090  0.028
1          0.278  0.395  0.259  0.068
2          0.073  0.220  0.445  0.261
3          0.018  0.068  0.240  0.674

Classifier evaluation complete. Continue to Stage 2...


In [ ]:
# ============================================================================
# 3. STAGE 2: CATEGORY-SPECIFIC REGRESSORS
# ============================================================================

print("\n" + "="*60)
print("STAGE 2: TRAINING CATEGORY-SPECIFIC REGRESSORS")
print("="*60)

category_regressors = {}
category_names = ['Short', 'Moderate', 'Long', 'Extreme']

for cat in range(4):
    cat_name = category_names[cat]
    print(f"\n{'-'*40}")
    print(f"Training regressor for '{cat_name}' stays (category {cat})")
    print(f"{'-'*40}")

    # Filter data for this category
    train_mask = y_cat_train == cat
    n_samples = train_mask.sum()

    print(f"Training samples: {n_samples:,}")

    if n_samples < 50:
        print(f"Skipping - too few samples")
        continue

    X_train_cat = X_train[train_mask]
    y_train_cat = y_days_train[train_mask]

    print(f"LOS range: {y_train_cat.min():.1f} - {y_train_cat.max():.1f} days")
    print(f"LOS mean: {y_train_cat.mean():.1f} days")

    # Adaptive hyperparameters based on category
    # Longer stays need more model capacity and less regularization
    depth_map = {0: 5, 1: 5, 2: 6, 3: 6}  # Reduced depths
    l2_map = {0: 3, 1: 3, 2: 2, 3: 2}
    iterations_map = {0: 300, 1: 400, 2: 500, 3: 500}  # Reduced iterations

    regressor = CatBoostRegressor(
        iterations=iterations_map[cat],
        learning_rate=0.1,  # Increased for faster convergence
        depth=depth_map[cat],

        loss_function='RMSE',
        eval_metric='MAE',

        cat_features=cat_features,
        l2_leaf_reg=l2_map[cat],

        task_type='CPU',
        thread_count=-1,

        early_stopping_rounds=50,
        verbose=False,
        random_state=42
    )

    # Create pool
    train_pool_reg = Pool(
        data=X_train_cat,
        label=y_train_cat,
        cat_features=cat_features
    )

    regressor.fit(train_pool_reg)
    category_regressors[cat] = regressor

    # Save regressor immediately after training
    regressor.save_model(f'regressor_cat{cat}_200k.cbm')
    print(f"Trained and saved as 'regressor_cat{cat}_200k.cbm'")

print(f"\n Trained {len(category_regressors)} category-specific regressors")


STAGE 2: TRAINING CATEGORY-SPECIFIC REGRESSORS

----------------------------------------
Training regressor for 'Short' stays (category 0)
----------------------------------------
Training samples: 62,627
LOS range: 1.0 - 2.0 days
LOS mean: 1.5 days
Trained and saved as 'regressor_cat0_200k.cbm'

----------------------------------------
Training regressor for 'Moderate' stays (category 1)
----------------------------------------
Training samples: 50,818
LOS range: 3.0 - 5.0 days
LOS mean: 3.7 days
Trained and saved as 'regressor_cat1_200k.cbm'

----------------------------------------
Training regressor for 'Long' stays (category 2)
----------------------------------------
Training samples: 26,235
LOS range: 6.0 - 10.0 days
LOS mean: 7.5 days
Trained and saved as 'regressor_cat2_200k.cbm'

----------------------------------------
Training regressor for 'Extreme' stays (category 3)
----------------------------------------
Training samples: 20,320
LOS range: 11.0 - 120.0 days
LOS mean: 

In [ ]:
# FIXED PREDICTION FUNCTIONS
def predict_hard_routing(X, y_cat_pred, category_regressors):
    """Route each sample to its predicted category's regressor"""
    predictions = np.zeros(len(X))
    y_cat_pred = np.array(y_cat_pred).ravel()

    for cat, regressor in category_regressors.items():
        mask = y_cat_pred == cat
        if mask.sum() > 0:
            if hasattr(X, 'iloc'):
                predictions[mask] = regressor.predict(X.iloc[mask])
            else:
                predictions[mask] = regressor.predict(X[mask])

    return predictions

def predict_soft_routing(X, y_cat_probs, category_regressors):
    """Weighted ensemble using classification probabilities"""
    predictions = np.zeros(len(X))

    for cat, regressor in category_regressors.items():
        cat_predictions = regressor.predict(X)
        predictions += cat_predictions * y_cat_probs[:, cat]

    return predictions

# Now generate predictions
print("="*60)
print("GENERATING PREDICTIONS")
print("="*60)

y_cat_pred_test = classifier.predict(X_test)
y_cat_probs_test = classifier.predict_proba(X_test)

print("Generating hard routing predictions...")
y_pred_hard = predict_hard_routing(X_test, y_cat_pred_test, category_regressors)

print("Generating soft routing predictions...")
y_pred_soft = predict_soft_routing(X_test, y_cat_probs_test, category_regressors)

print("Predictions complete!")

GENERATING PREDICTIONS
Generating hard routing predictions...
Generating soft routing predictions...
Predictions complete!


In [ ]:
# ============================================================================
# 5. EVALUATION
# ============================================================================

def evaluate_predictions(y_true, y_pred, y_cat_true, method_name):
    """Comprehensive evaluation of predictions"""
    print(f"\n{'='*60}")
    print(f"EVALUATION: {method_name}")
    print(f"{'='*60}")

    # Overall metrics
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"\nOverall Performance:")
    print(f"  MAE:  {mae:.2f} days")
    print(f"  RMSE: {rmse:.2f} days")
    print(f"  R²:   {r2:.3f}")

    # Performance by category
    print(f"\nPerformance by Category:")
    print(f"{'Category':<12} {'Count':<8} {'MAE':<8} {'RMSE':<8} {'R²':<8} {'Mean True':<12} {'Mean Pred':<12}")
    print("-" * 90)

    for cat in range(4):
        cat_name = category_names[cat]
        mask = y_cat_true == cat

        if mask.sum() == 0:
            continue

        mae_cat = mean_absolute_error(y_true[mask], y_pred[mask])
        rmse_cat = np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))
        r2_cat = r2_score(y_true[mask], y_pred[mask])
        mean_true = y_true[mask].mean()
        mean_pred = y_pred[mask].mean()

        print(f"{cat_name:<12} {mask.sum():<8} {mae_cat:<8.2f} {rmse_cat:<8.2f} "
              f"{r2_cat:<8.3f} {mean_true:<12.2f} {mean_pred:<12.2f}")

    # Performance on long stays (the critical part!)
    print(f"\nPerformance on Long Stays:")

    thresholds = [7, 14, 21, 30]
    for threshold in thresholds:
        mask = y_true > threshold
        if mask.sum() > 0:
            mae_long = mean_absolute_error(y_true[mask], y_pred[mask])
            mean_true = y_true[mask].mean()
            mean_pred = y_pred[mask].mean()
            print(f"  LOS > {threshold:2d} days (n={mask.sum():5d}): "
                  f"MAE={mae_long:6.2f}, True={mean_true:6.2f}, Pred={mean_pred:6.2f}")

    return mae, rmse, r2

# Evaluate both methods
mae_hard, rmse_hard, r2_hard = evaluate_predictions(
    y_days_test, y_pred_hard, y_cat_test, "HARD ROUTING"
)

mae_soft, rmse_soft, r2_soft = evaluate_predictions(
    y_days_test, y_pred_soft, y_cat_test, "SOFT ROUTING (RECOMMENDED)"
)


EVALUATION: HARD ROUTING

Overall Performance:
  MAE:  3.70 days
  RMSE: 7.28 days
  R²:   0.283

Performance by Category:
Category     Count    MAE      RMSE     R²       Mean True    Mean Pred   
------------------------------------------------------------------------------------------
Short        15656    1.80     4.00     -63.459  1.55         3.09        
Moderate     12705    2.86     5.14     -42.136  3.72         5.28        
Long         6559     5.08     7.54     -29.564  7.48         9.76        
Extreme      5080     9.90     15.12    0.118    21.54        18.28       

Performance on Long Stays:
  LOS >  7 days (n= 7909): MAE=  8.47, True= 17.00, Pred= 15.76
  LOS > 14 days (n= 3007): MAE= 11.62, True= 27.90, Pred= 20.98
  LOS > 21 days (n= 1478): MAE= 16.54, True= 38.61, Pred= 25.03
  LOS > 30 days (n=  715): MAE= 25.58, True= 52.65, Pred= 29.39

EVALUATION: SOFT ROUTING (RECOMMENDED)

Overall Performance:
  MAE:  3.41 days
  RMSE: 6.61 days
  R²:   0.410

Performance b

In [ ]:
# Evaluate classifier (fixed version)
y_cat_pred = classifier.predict(X_test)
y_cat_probs = classifier.predict_proba(X_test)

# Convert to numpy arrays
y_cat_test_array = np.array(y_cat_test)
y_cat_pred_array = np.array(y_cat_pred).ravel()

print("="*60)
print("CLASSIFIER PERFORMANCE")
print("="*60)

kappa = cohen_kappa_score(y_cat_test_array, y_cat_pred_array, weights='quadratic')
print(f"Quadratic Weighted Kappa: {kappa:.3f}")
print(f"Accuracy: {(y_cat_test_array == y_cat_pred_array).mean():.3f}")

print("\nClassification Report:")
print(classification_report(y_cat_test_array, y_cat_pred_array,
                            target_names=['Short', 'Moderate', 'Long', 'Extreme'],
                            digits=3))

# Confusion Matrix
cm = pd.crosstab(y_cat_test_array, y_cat_pred_array,
                 rownames=['Actual'], colnames=['Predicted'],
                 normalize='index')
print("\nConfusion Matrix (Row-Normalized): ")
print(cm.round(3))

CLASSIFIER PERFORMANCE
Quadratic Weighted Kappa: 0.651
Accuracy: 0.544

Classification Report:
              precision    recall  f1-score   support

       Short      0.717     0.663     0.689     15656
    Moderate      0.491     0.395     0.438     12705
        Long      0.330     0.445     0.379      6559
     Extreme      0.531     0.674     0.594      5080

    accuracy                          0.544     40000
   macro avg      0.517     0.544     0.525     40000
weighted avg      0.558     0.544     0.546     40000


Confusion Matrix (Row-Normalized): 
Predicted      0      1      2      3
Actual                               
0          0.663  0.218  0.090  0.028
1          0.278  0.395  0.259  0.068
2          0.073  0.220  0.445  0.261
3          0.018  0.068  0.240  0.674


In [ ]:
# ============================================================================
# 6. FEATURE IMPORTANCE
# ============================================================================

print("\n" + "="*60)
print("TOP FREATURES FOR CLASSIFIER")
print("="*60)

feature_importance = classifier.get_feature_importance()
feature_names = classifier.get_feature_importance(train_pool_clf)
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print(importance_df.head(30).to_string(index=False))


TOP FREATURES FOR CLASSIFIER
  feature  importance
20.135080   20.102190
14.208335   13.938616
13.920731   13.005750
 9.159137    9.270982
 7.713553    7.620474
 7.031738    7.054577
 6.131039    6.354734
 5.652476    5.431847
 3.164750    3.541488
 2.950608    2.938022
 2.571113    2.684154
 1.877771    1.956800
 1.485991    1.712548
 1.490064    1.593223
 1.197677    1.384191
 0.722311    0.873772
 0.288614    0.279158
 0.299014    0.257473


In [ ]:
# ============================================================================
# 7. SAVE MODELS FOR FULL DATASET TRAINING
# ============================================================================

print("\n" + "="*60)
print("SAVING MODELS")
print("="*60)

# Save models
classifier.save_model('classifier_200k.cbm')
for cat, regressor in category_regressors.items():
    regressor.save_model(f'regressor_cat{cat}_200k.cbm')

print("✓ Models saved:")
print("  - classifier_200k.cbm")
for cat in category_regressors.keys():
    print(f"  - regressor_cat{cat}_200k.cbm")


SAVING MODELS
✓ Models saved:
  - classifier_200k.cbm
  - regressor_cat0_200k.cbm
  - regressor_cat1_200k.cbm
  - regressor_cat2_200k.cbm
  - regressor_cat3_200k.cbm


In [ ]:
print("\n" + "="*60)
print("RECOMMENDATIONS")
print("="*60)

if mae_soft < mae_hard:
  print("\nRecommended strategy: SOFT ROUTING (Probability weighted ensemble)")
  print(f"  It improves MAE by {mae_hard - mae_soft:.2f} days")
else:
  print("\nRecommended strategy: HARD ROUTING (direct category assignment)")
  print(f"  It improves MAE by {mae_soft - mae_hard:.2f} days")

if kappa < 0.65:
  print("\nClassifier could be improved: ")
  print("   - Try more iterations")
  print("   - Increase depth to 12")
  print("   - Create feature interactions")

# Check long stay performance
long_mask = y_days_test > 14
if long_mask.sum() > 0:
  mae_long = mean_absolute_error(y_days_test[long_mask], y_pred_soft[long_mask])
  if mae_long > 7:
    print("\n Long stay predictions need improvement:")
    print("   - Consider addping sample weights")
    print("   - Try quantile regression for category 3 (extreme)")
    print("   - Add feature engineering")


RECOMMENDATIONS

Recommended strategy: SOFT ROUTING (Probability weighted ensemble)
  It improves MAE by 0.29 days

 Long stay predictions need improvement:
   - Consider addping sample weights
   - Try quantile regression for category 3 (extreme)
   - Add feature engineering


In [ ]:
import numpy as np
from catboost import CatBoostRegressor, Pool

# ============================================================================
# FUNCTION TO CREATE SAMPLE WEIGHTS
# ============================================================================

def create_sample_weights(y, method='log_scaled'):
  """
  Create sample weights that emphasize longer stays

  Parameters:
  -----------
  y : array-like
      Length of stay values
  method : str
        'log_scaled' - logarithmic weighting
        'sqrt_scaled' - square root weighting
        'inverse_frequency' - bin-based inverse frequency
  Returns:
  -------
  weights : numpy array
      Sample weights (normalized to mean=1)
  """

  y = np.array(y)

  if method == 'log_scaled':
    weights = np.log1p (y)
    weights = weights / weights.mean()

  elif method == 'sqrt_scaled':
    # less aggressive than log
    weights = np.sqrt(y)
    weights = weights / weights.mean()

  elif method == 'inverse_frequency':
    #Bin stays and weight by inverse frequency
    bins = np.percentile(y, [0, 25, 50, 75, 100])
    bin_labels = np.digitize(y, bins[:-1]) - 1
    bin_counts = np.bincount(bin_labels)
    bin_weights = 1 / (bin_counts + 1) # +1 to avoid division by 0
    weights = bin_weights[bin_labels]
    weights = weights / weights.mean()

  return weights

# ============================================================================
# TRAIN CATEGORY-SPECIFIC REGRESSORS WITH SAMPLE WEIGHTS
# ============================================================================
print("="*60)
print("TRAINING WEIGHTED REGRESSORS")
print("="*60)

weighted_regressors = {}
category_names = ['Short', 'Moderate', 'Long', 'Extreme']

for cat in range(4):
    cat_name = category_names[cat]
    print(f"\n{'-'*40}")
    print(f"Training WEIGHTED regressor for '{cat_name}' (category {cat})")
    print(f"{'-'*40}")

    # Filter data for this category
    train_mask = y_cat_train == cat
    n_samples = train_mask.sum()

    print(f"Training samples: {n_samples:,}")

    if n_samples < 50:
        print(f"Skipping - too few samples")
        continue

    X_train_cat = X_train[train_mask]
    y_train_cat = y_days_train[train_mask]

    print(f"LOS range: {y_train_cat.min():.1f} - {y_train_cat.max():.1f} days")
    print(f"LOS mean: {y_train_cat.mean():.1f} days")
    print(f"LOS median: {np.median(y_train_cat):.1f} days")

    # CREATE SAMPLE WEIGHTS
    # Use log_scaled for all categories - emphasizes longer stays
    sample_weights = create_sample_weights(y_train_cat, method='log_scaled')

    print(f"Weight range: {sample_weights.min():.2f} - {sample_weights.max():.2f}")
    print(f"Weight mean: {sample_weights.mean():.2f}")

    # Adaptive hyperparameters based on category
    depth_map = {0: 5, 1: 5, 2: 6, 3: 7}  # Slightly deeper for extreme
    l2_map = {0: 3, 1: 3, 2: 2, 3: 1}  # Less regularization for extreme
    iterations_map = {0: 300, 1: 400, 2: 500, 3: 600}  # More iterations for extreme

    regressor = CatBoostRegressor(
        iterations=iterations_map[cat],
        learning_rate=0.1,
        depth=depth_map[cat],

        loss_function='RMSE',
        eval_metric='MAE',

        cat_features=cat_features,
        l2_leaf_reg=l2_map[cat],

        task_type='CPU',
        thread_count=-1,

        early_stopping_rounds=50,
        verbose=False,
        random_state=42
    )

    # Create pool WITH sample weights
    train_pool_reg = Pool(
        data=X_train_cat,
        label=y_train_cat,
        cat_features=cat_features,
        weight=sample_weights  # KEY: Add weights here!
    )

    regressor.fit(train_pool_reg)
    weighted_regressors[cat] = regressor

    # Save weighted regressor
    regressor.save_model(f'regressor_cat{cat}_weighted_200k.cbm')
    print(f"✓ Trained and saved as 'regressor_cat{cat}_weighted_200k.cbm'")

print(f"\n✓ Trained {len(weighted_regressors)} weighted regressors")

# ============================================================================
# GENERATE PREDICTIONS WITH WEIGHTED REGRESSORS
# ============================================================================

print("\n" + "="*60)
print("GENERATING WEIGHTED PREDICTIONS")
print("="*60)

def predict_hard_routing(X, y_cat_pred, category_regressors):
    """Route each sample to its predicted category's regressor"""
    predictions = np.zeros(len(X))
    y_cat_pred = np.array(y_cat_pred).ravel()

    for cat, regressor in category_regressors.items():
        mask = y_cat_pred == cat
        if mask.sum() > 0:
            if hasattr(X, 'iloc'):
                predictions[mask] = regressor.predict(X.iloc[mask])
            else:
                predictions[mask] = regressor.predict(X[mask])

    return predictions

def predict_soft_routing(X, y_cat_probs, category_regressors):
    """Weighted ensemble using classification probabilities"""
    predictions = np.zeros(len(X))

    for cat, regressor in category_regressors.items():
        cat_predictions = regressor.predict(X)
        predictions += cat_predictions * y_cat_probs[:, cat]

    return predictions

y_cat_pred_test = classifier.predict(X_test)
y_cat_probs_test = classifier.predict_proba(X_test)

print("Generating hard routing predictions...")
y_pred_hard_weighted = predict_hard_routing(X_test, y_cat_pred_test, weighted_regressors)

print("Generating soft routing predictions...")
y_pred_soft_weighted = predict_soft_routing(X_test, y_cat_probs_test, weighted_regressors)

print("✓ Weighted predictions complete!")

# ============================================================================
# EVALUATION FUNCTION
# ============================================================================

def evaluate_predictions(y_true, y_pred, y_cat_true, method_name):
    """Comprehensive evaluation of predictions"""
    print(f"\n{'='*60}")
    print(f"EVALUATION: {method_name}")
    print(f"{'='*60}")

    # Convert to numpy arrays
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_cat_true = np.array(y_cat_true)

    # Overall metrics
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"\nOverall Performance:")
    print(f"  MAE:  {mae:.2f} days")
    print(f"  RMSE: {rmse:.2f} days")
    print(f"  R²:   {r2:.3f}")

    # Performance by category
    print(f"\nPerformance by Category:")
    print(f"{'Category':<12} {'Count':<8} {'MAE':<8} {'RMSE':<8} {'R²':<8} {'Mean True':<12} {'Mean Pred':<12}")
    print("-" * 90)

    for cat in range(4):
        cat_name = ['Short', 'Moderate', 'Long', 'Extreme'][cat]
        mask = y_cat_true == cat

        if mask.sum() == 0:
            continue

        mae_cat = mean_absolute_error(y_true[mask], y_pred[mask])
        rmse_cat = np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))
        r2_cat = r2_score(y_true[mask], y_pred[mask])
        mean_true = y_true[mask].mean()
        mean_pred = y_pred[mask].mean()

        print(f"{cat_name:<12} {mask.sum():<8} {mae_cat:<8.2f} {rmse_cat:<8.2f} "
              f"{r2_cat:<8.3f} {mean_true:<12.2f} {mean_pred:<12.2f}")

    # Performance on long stays (the critical part!)
    print(f"\nPerformance on Long Stays:")

    thresholds = [7, 14, 21, 30]
    for threshold in thresholds:
        mask = y_true > threshold
        if mask.sum() > 0:
            mae_long = mean_absolute_error(y_true[mask], y_pred[mask])
            mean_true = y_true[mask].mean()
            mean_pred = y_pred[mask].mean()
            print(f"  LOS > {threshold:2d} days (n={mask.sum():5d}): "
                  f"MAE={mae_long:6.2f}, True={mean_true:6.2f}, Pred={mean_pred:6.2f}")

    return mae, rmse, r2

# ============================================================================
# EVALUATE WEIGHTED PREDICTIONS
# ============================================================================

print("\n" + "="*60)
print("COMPARING: ORIGINAL vs WEIGHTED REGRESSORS")
print("="*60)

mae_hard_weighted, rmse_hard_weighted, r2_hard_weighted = evaluate_predictions(
    y_days_test, y_pred_hard_weighted, y_cat_test, "WEIGHTED - HARD ROUTING"
)

mae_soft_weighted, rmse_soft_weighted, r2_soft_weighted = evaluate_predictions(
    y_days_test, y_pred_soft_weighted, y_cat_test, "WEIGHTED - SOFT ROUTING"
)

# ============================================================================
# COMPARISON SUMMARY
# ============================================================================

print("\n" + "="*60)
print("IMPROVEMENT SUMMARY")
print("="*60)

print("\nOverall MAE Comparison:")
print(f"  Original Soft Routing:  3.41 days")
print(f"  Weighted Soft Routing:  {mae_soft_weighted:.2f} days")
print(f"  Improvement:            {3.41 - mae_soft_weighted:.2f} days")

print("\nLong Stay (>14 days) Performance:")
long_mask = np.array(y_days_test) > 14
if long_mask.sum() > 0:
    from sklearn.metrics import mean_absolute_error
    mae_long_weighted = mean_absolute_error(
        np.array(y_days_test)[long_mask],
        y_pred_soft_weighted[long_mask]
    )
    print(f"  Original:  12.40 days MAE")
    print(f"  Weighted:  {mae_long_weighted:.2f} days MAE")
    print(f"  Improvement: {12.40 - mae_long_weighted:.2f} days")

TRAINING WEIGHTED REGRESSORS

----------------------------------------
Training WEIGHTED regressor for 'Short' (category 0)
----------------------------------------
Training samples: 62,627
LOS range: 1.0 - 2.0 days
LOS mean: 1.5 days
LOS median: 2.0 days
Weight range: 0.76 - 1.20
Weight mean: 1.00
✓ Trained and saved as 'regressor_cat0_weighted_200k.cbm'

----------------------------------------
Training WEIGHTED regressor for 'Moderate' (category 1)
----------------------------------------
Training samples: 50,818
LOS range: 3.0 - 5.0 days
LOS mean: 3.7 days
LOS median: 4.0 days
Weight range: 0.90 - 1.16
Weight mean: 1.00
✓ Trained and saved as 'regressor_cat1_weighted_200k.cbm'

----------------------------------------
Training WEIGHTED regressor for 'Long' (category 2)
----------------------------------------
Training samples: 26,235
LOS range: 6.0 - 10.0 days
LOS mean: 7.5 days
LOS median: 7.0 days
Weight range: 0.91 - 1.13
Weight mean: 1.00
✓ Trained and saved as 'regressor_cat2_

##**Trying to Engineer Features**
- Create interactions

In [46]:
from catboost import CatBoostClassifier, CatBoostRegressor, Pool
import pandas as pd
import numpy as np

# Define the persistent path for your models.
# Make sure your original models are saved here!
DRIVE_MODEL_PATH = '/content/drive/MyDrive/Sparcs_Datafiles/'

# ============================================================================
# LOAD ORIGINAL MODELS (200k)
# ============================================================================

# Load classifier
print("Loading classifier (200k)...")
classifier = CatBoostClassifier()
# FIX: Use the full Google Drive path for the model file
classifier.load_model('/content/drive/MyDrive/classifier_200k.cbm')

# Load regressors
print("Loading 4 regressors (200k)...")
category_regressors = {}
for cat in range(4):
    regressor = CatBoostRegressor()
    # FIX: Use the full Google Drive path for the model file
    regressor.load_model('/content/drive/MyDrive/' + f'regressor_cat{cat}_200k.cbm')
    category_regressors[cat] = regressor

print("Original models loaded successfully.")

Loading classifier (200k)...
Loading 4 regressors (200k)...
Original models loaded successfully.


In [47]:
# 1. Define engineer function
def engineer_interactions(df):
  df_eng = df.copy()

  # Severity + Diagnosis (interaction top feature #2 and #7)
  df_eng['severity_dx'] = df_eng['apr_severity_code'].astype(str) + '_' + df_eng['ccsr_dx_code'].astype(str)

  # Procedure + DRG (interaction of top feature #3 and #1)
  df_eng['px_drg'] = df_eng['ccsr_px_code'].astype(str) + '_' + df_eng['apr_drg_code'].astype(str)

  # High risk flag (explicit boolean)
  high_severity = df_eng['apr_severity_code'].astype(str).isin(['3', '4'])
  high_mortality = df_eng['apr_mortality_risk'].astype(str).isin(['3', '4'])
  df_eng['is_high_risk'] = (high_severity & high_mortality).astype(int)

  return df_eng

# 2. Apply to dataframe
print ("Engineering features...")
df_engineered = engineer_interactions(df)

# 3. Update categorical feature list
# Add two new string columns to the list of cat features
cat_features_eng = cat_features + ['severity_dx', 'px_drg']

# 4. Update feature columns list
feature_cols_eng = cat_features_eng + ['num_payment_types', 'is_high_risk']

# 5. Create new X and y
X_eng = df_engineered[feature_cols_eng].copy()
# y_days and y_category same

# 6. Resplit data (need to include the new cols)
print("Splitting data...")
X_train_eng, X_test_eng, y_cat_train, y_cat_test, y_days_train, y_days_test = train_test_split(
    X_eng, y_category, y_days,
    test_size=0.2, stratify=y_category, random_state=42
)

print(f"New feature count: {X_train_eng.shape[1]} (was {X_train.shape[1]})")

Engineering features...
Splitting data...
New feature count: 21 (was 18)


In [48]:
# ============================================================================
# RETRAIN CLASSIFIER
# ============================================================================

print("\n" + "="*60)
print("RETRAINING CLASSIFIER WITH NEW FEATURES")
print("="*60)

train_pool_clf = Pool(X_train_eng, y_cat_train, cat_features=cat_features_eng)
test_pool_clf = Pool(X_test_eng, y_cat_test, cat_features=cat_features_eng)

classifier_eng = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    cat_features=cat_features_eng,  # Use new list
    auto_class_weights='Balanced',
    eval_metric='WKappa',
    task_type='CPU',
    verbose=100
)

classifier_eng.fit(train_pool_clf, eval_set=test_pool_clf)

# ============================================================================
# RETRAIN REGRESSORS
# ============================================================================

print("\n" + "="*60)
print("RETRAINING REGRESSORS WITH NEW FEATURES")
print("="*60)

regressors_eng = {}
# Get predicted probs for soft routing
y_cat_probs_eng = classifier_eng.predict_proba(X_test_eng)

for cat in range(4):
  print(f"Training regressor for Category {cat}...")

  # Filter data
  mask = y_cat_train == cat
  X_train_cat = X_train_eng[mask]
  y_train_cat = y_days_train[mask]

  # create pool
  pool = Pool(X_train_cat, y_train_cat, cat_features=cat_features_eng)

  # Define model
  reg = CatBoostRegressor(
    iterations=500,
    learning_rate=0.1,
    depth=6 if cat >= 2 else 5,   # Deeper for longer stays
    loss_function='RMSE',
    cat_features=cat_features_eng,
    task_type='CPU',
    verbose=False
  )

  reg.fit(pool)
  regressors_eng[cat] = regressor

print("Retraining complete!")


RETRAINING CLASSIFIER WITH NEW FEATURES
0:	learn: 0.6288647	test: 0.6393767	best: 0.6393767 (0)	total: 4.64s	remaining: 38m 35s
100:	learn: 0.6809641	test: 0.6884670	best: 0.6884670 (100)	total: 7m 22s	remaining: 29m 9s
200:	learn: 0.6885086	test: 0.6928487	best: 0.6931905 (198)	total: 15m 4s	remaining: 22m 25s
300:	learn: 0.6923248	test: 0.6935975	best: 0.6938055 (270)	total: 22m 25s	remaining: 14m 49s
400:	learn: 0.6956873	test: 0.6935379	best: 0.6945674 (367)	total: 29m 40s	remaining: 7m 19s
499:	learn: 0.6980998	test: 0.6940654	best: 0.6945674 (367)	total: 36m 56s	remaining: 0us

bestTest = 0.6945673529
bestIteration = 367

Shrink model to first 368 iterations.

RETRAINING REGRESSORS WITH NEW FEATURES
Training regressor for Category 0...
Training regressor for Category 1...
Training regressor for Category 2...
Training regressor for Category 3...
Retraining complete!


In [52]:
# ============================================================================
# RETRAIN REGRESSORS (CORRECTED ASSIGNMENT)
# ============================================================================

print("\n" + "="*60)
print("RETRAINING REGRESSORS WITH NEW FEATURES")
print("="*60)

regressors_eng = {}
# Get predicted probs for soft routing (This is fine here)
y_cat_probs_eng = classifier_eng.predict_proba(X_test_eng)

for cat in range(4):
  print(f"Training regressor for Category {cat}...")

  # Filter data (Correct)
  mask = y_cat_train == cat
  X_train_cat = X_train_eng[mask]
  y_train_cat = y_days_train[mask]

  # create pool (Correct)
  pool = Pool(X_train_cat, y_train_cat, cat_features=cat_features_eng)

  # Define model (Correct)
  reg = CatBoostRegressor(
    iterations=500,
    learning_rate=0.1,
    depth=6 if cat >= 2 else 5,
    loss_function='RMSE',
    cat_features=cat_features_eng,
    task_type='CPU',
    verbose=False
  )

  reg.fit(pool)
  regressors_eng[cat] = reg # <-- FIXED: Assign the fitted model 'reg'

print("Retraining complete!")


RETRAINING REGRESSORS WITH NEW FEATURES
Training regressor for Category 0...
Training regressor for Category 1...
Training regressor for Category 2...
Training regressor for Category 3...
Retraining complete!


In [53]:
# save models
classifier_eng.save_model('/content/drive/MyDrive/Sparcs_Datafiles/classifier_eng.cbm')
for cat, regressor in regressors_eng.items():
  regressor.save_model(f'/content/drive/MyDrive/Sparcs_Datafiles/regressor_cat{cat}_eng.cbm')

In [54]:
# ============================================================================
# PREDICTION AND EVALUATION
# ============================================================================
from catboost import Pool
from sklearn.metrics import mean_absolute_error, r2_score

# Helper for soft routing - MODIFIED to accept the categorical feature list
def predict_soft_routing_eng(X, probs, regressors, cat_features):
    preds = np.zeros(len(X))
    # Create a single Pool for prediction to ensure CatBoost recognizes
    # all categorical features like 'severity_dx' and 'px_drg'.
    prediction_pool = Pool(X, cat_features=cat_features)

    for cat, reg in regressors.items():
      # Predict using the robust Pool object
      preds += reg.predict(prediction_pool) * probs[:, cat]
    return preds

# Generate predictions
print("\nEvaluating...")

# Pass the categorical feature list (cat_features_eng) to the function
y_pred_eng = predict_soft_routing_eng(X_test_eng, y_cat_probs_eng, regressors_eng, cat_features_eng)

# Calculate MAE
# FIX: Ensure you are using the correct target variable for the engineered test set (y_days_test)
mae_new = mean_absolute_error(y_days_test, y_pred_eng)
r2_new = r2_score(y_days_test, y_pred_eng)

print(f"\nRESULTS COMPARISON:")
print(f"Original MAE: 3.41")
print(f"Weighted MAE: 3.56")
print(f"Feature Eng MAE: {mae_new:.2f}")

print(f"\nR² Score (Target: > 0.41): {r2_new:.3f}")


Evaluating...

RESULTS COMPARISON:
Original MAE: 3.41
Weighted MAE: 3.56
Feature Eng MAE: 3.38

R² Score (Target: > 0.41): 0.411
